# **Symmetry Reduction in the Maximum Cut Problem: ILP Formulation**

This notebook presents an Integer Linear Programming formulation for the maximum cut problem and investigates the role of flip symmetry. We demonstrate how the inherent two-way labeling of graph partitions generates pairs of equivalent optimal solutions and how a single fixing constraint can eliminate this redundancy while preserving optimality.

---

## *Mathematical Background*

### **Maximum Cut as Integer Linear Program**

Given an undirected graph $G = (V, E)$ with edge weights $w_e > 0$, the maximum cut problem seeks a partition $V = S \cup \bar{S}$ (with $S \cap \bar{S} = \emptyset$) that maximizes the total weight of edges crossing the cut, that is, edges with one endpoint in $S$ and the other in $\bar{S}$.<sup>[[1]](#ref1)</sup>

We introduce two families of binary decision variables: $z_v \in \{0, 1\}$ for each vertex $v \in V$, indicating the side of the partition ($z_v = 0$ if $v \in S$, $z_v = 1$ if $v \in \bar{S}$); and $y_{uv} \in \{0, 1\}$ for each edge $\{u,v\} \in E$, equal to $1$ if and only if the edge crosses the cut.

The objective function maximizes the total weight of cut edges:

\begin{equation*}
\max \sum_{\{u,v\} \in E} w_{uv} \cdot y_{uv}
\end{equation*}

The condition $y_{uv} = 1 \iff z_u \neq z_v$ is nonlinear, but can be linearized by the following four families of constraints:

\begin{equation*}
\begin{aligned}
y_{uv} &\leq z_u + z_v \\
y_{uv} &\leq 2 - z_u - z_v \\
y_{uv} &\geq z_u - z_v \\
y_{uv} &\geq z_v - z_u
\end{aligned}
\quad \forall \{u,v\} \in E
\end{equation*}

One can verify that these four inequalities jointly force $y_{uv} = 0$ when $z_u = z_v$ and $y_{uv} = 1$ when $z_u \neq z_v$, correctly encoding the cut indicator. Combined with the binary domains and the maximization objective, this ILP yields an exact solution to the maximum cut problem.<sup>[[2]](#ref2)</sup>

---

### **Graph Symmetries**

The maximum cut problem exhibits two sources of symmetry that generate equivalent optimal solutions.<sup>[[2]](#ref2), [[3]](#ref3)</sup>

The first and most fundamental is **flip symmetry**. For any partition $(S, \bar{S})$, the complementary partition $(\bar{S}, S)$ cuts exactly the same set of edges and therefore achieves the same objective value. Formally, if $\mathbf{z}$ is an optimal solution, then the complement $\bar{\mathbf{z}}$ defined by $\bar{z}_v = 1 - z_v$ for all $v \in V$ is also optimal:

\begin{equation*}
\sum_{\{u,v\} \in E} w_{uv} \cdot y_{uv}(\mathbf{z}) = \sum_{\{u,v\} \in E} w_{uv} \cdot y_{uv}(\bar{\mathbf{z}})
\end{equation*}

This symmetry is universal: it is present in every instance of the maximum cut problem, regardless of the graph structure or edge weights, since the partition labels $0$ and $1$ are interchangeable. The group generated by this transformation is $\mathbb{Z}_2$, which acts on the solution space by mapping each solution to its complement. Every feasible solution belongs to an orbit of size exactly $2$ under this action.

The second source is **vertex symmetry**, arising from graph automorphisms. A bijection $\pi: V \to V$ that preserves adjacency and edge weights maps any feasible partition to another feasible partition with the same cut value, generating additional equivalent solutions beyond those produced by the flip.

---

### **Symmetry Breaking**

To eliminate the duplicate solutions generated by flip symmetry, we observe that for any optimal partition $(S, \bar{S})$, exactly one of the two flips has vertex $1$ in $S$ (i.e., $z_1 = 0$) and the other has vertex $1$ in $\bar{S}$ (i.e., $z_1 = 1$). We therefore add the single fixing constraint:<sup>[[2]](#ref2), [[3]](#ref3)</sup>

\begin{equation*}
z_1 = 0
\end{equation*}

This constraint fixes vertex $1$ to the side $S$ in all feasible solutions, eliminating the flip of every partition while retaining exactly one canonical representative per orbit.

A valid symmetry-breaking scheme must satisfy two properties:<sup>[[3]](#ref3)</sup>
- *Completeness*: for any optimal partition, its canonical representative (the flip with $z_1 = 0$) satisfies the constraint and therefore remains feasible.
- *Soundness*: the canonical representative has the same objective value as its flip, since the cut value is invariant under the flip transformation.

The constraint eliminates exactly one of the two solutions in every flip orbit, reducing the solution count by a factor of $|\mathbb{Z}_2| = 2$.

---

### **References**

<a id="ref1"></a>
<sup>[[1]](#ref1)</sup> Garey, M. R., & Johnson, D. S. (1979). *Computers and Intractability: A Guide to the Theory of NP-Completeness*. W. H. Freeman.

<a id="ref2"></a>
<sup>[[2]](#ref2)</sup> Margot, F. (2010). Symmetry in Integer Linear Programming. In *50 Years of Integer Programming 1958–2008* (pp. 647–686). Springer.

<a id="ref3"></a>
<sup>[[3]](#ref3)</sup> Liberti, L. (2012). Symmetry in Mathematical Programming. In *Mixed Integer Nonlinear Programming* (pp. 263–283). Springer.

## *Computational Implementation*

In [1]:
using JuMP
using HiGHS
println("Packages loaded | Julia ", VERSION)

Packages loaded | Julia 1.11.5


In [2]:
function build_mc_model(edges, weights, N;
                        symmetry_breaking=false, fixed_obj=nothing)
    """
    Builds an integer linear programming model for the maximum cut problem.

    Args:
        edges: List of edges (tuples (u,v) with u < v) of the graph
        weights: Dictionary with the weight of each edge
        N: Total number of vertices in the graph
        symmetry_breaking: (optional) Enables the fixing constraint z[1] = 0
        fixed_obj: (optional) Fixes the objective value (for enumerating optimal solutions)

    Returns: (model, z, y) - The JuMP model and decision variables
    """

    model = Model(HiGHS.Optimizer)
    set_silent(model)

    @variable(model, z[1:N], Bin)
    @variable(model, y[(u,v) in edges], Bin)

    @objective(model, Max, sum(weights[(u,v)] * y[(u,v)] for (u,v) in edges))

    # Linearization of y_{uv} = z_u XOR z_v
    for (u,v) in edges
        @constraint(model, y[(u,v)] <= z[u] + z[v])
        @constraint(model, y[(u,v)] <= 2 - z[u] - z[v])
        @constraint(model, y[(u,v)] >= z[u] - z[v])
        @constraint(model, y[(u,v)] >= z[v] - z[u])
    end

    # Fix objective value for solution enumeration
    if fixed_obj !== nothing
        @constraint(model, sum(weights[(u,v)] * y[(u,v)] for (u,v) in edges) == fixed_obj)
    end

    # Symmetry breaking: fix vertex 1 to side S
    if symmetry_breaking
        @constraint(model, z[1] == 0)
    end

    return model, z, y
end

function extract_cut(z, N)
    """
    Extracts the partition from the binary decision variables.

    Args:
        z: Model decision variables (partition assignment)
        N: Number of vertices

    Returns: (S, Sbar) - the two sides of the partition as integer vectors
    """

    S    = [v for v in 1:N if value(z[v]) < 0.5]
    Sbar = [v for v in 1:N if value(z[v]) > 0.5]
    return S, Sbar
end

function enumerate_cuts(edges, weights, N, opt_val;
                        symmetry_breaking=false)
    """
    Enumerates all optimal cuts (up to 100) with the maximum cut value.

    Args:
        edges: List of graph edges
        weights: Dictionary with edge weights
        N: Total number of vertices
        opt_val: Optimal value (maximum cut weight) to match
        symmetry_breaking: (optional) Enables fixing constraint z[1] = 0

    Returns: Vector of cuts (each cut is a tuple (S, Sbar) of integer vectors)
    """

    solutions  = Vector{Tuple{Vector{Int}, Vector{Int}}}()
    z_vectors  = Vector{Vector{Int}}()

    for _ in 1:100
        model, z, y = build_mc_model(edges, weights, N,
                                      symmetry_breaking=symmetry_breaking,
                                      fixed_obj=opt_val)

        # No-good cuts: the new solution must differ from every previous one
        # in at least one vertex assignment
        for prev_z in z_vectors
            @constraint(model,
                sum(    z[v] for v in 1:N if prev_z[v] == 0; init=0) +
                sum(1 - z[v] for v in 1:N if prev_z[v] == 1; init=0) >= 1)
        end

        optimize!(model)
        termination_status(model) != OPTIMAL && break

        S, Sbar = extract_cut(z, N)
        push!(solutions, (S, Sbar))
        push!(z_vectors, [round(Int, value(z[v])) for v in 1:N])
    end

    return solutions
end

enumerate_cuts (generic function with 1 method)

### **Example**: Complete Graph $K_4$

#### Problem Instance

Consider the complete graph $G = K_4$ with four vertices, where every pair of vertices is connected by an edge:

\begin{equation*}
\begin{array}{ccccc}
[1] & \text{---} & & \text{---} & [2] \\
|\!\diagdown & & & & \diagup\!| \\
| & & \times & & | \\
|\!\diagup & & & & \diagdown\!| \\
[4] & \text{---} & & \text{---} & [3]
\end{array}
\end{equation*}

where $\times$ denotes the crossing diagonals $\{1,3\}$ and $\{2,4\}$. The vertex set is $V = \{1,2,3,4\}$ and the edge set is $E = \{\{1,2\},\{1,3\},\{1,4\},\{2,3\},\{2,4\},\{3,4\}\}$, giving $|E| = 6$ edges, all with unit weight $w_e = 1$.

For $K_4$, the maximum cut value and the number of optimal partitions can be determined analytically. Any partition of the four vertices into two sides of sizes $|S|$ and $|\bar{S}|$ cuts exactly $|S| \cdot |\bar{S}|$ edges (since every cross-pair contributes one edge in $K_4$). This product is maximized by a balanced partition:

\begin{equation*}
\max_{|S|+|\bar{S}|=4} |S| \cdot |\bar{S}| = 2 \cdot 2 = 4
\end{equation*}

achieved uniquely by balanced $2$-$2$ partitions. The number of such partitions is $\binom{4}{2} / 2 = 3$ essentially distinct splits, each appearing twice due to the flip symmetry (swapping $S$ and $\bar{S}$), yielding $3 \times 2 = 6$ optimal solutions in total.

The graph possesses a rich vertex symmetry: $K_4$ is vertex-transitive and edge-transitive, with automorphism group $\mathrm{Aut}(K_4) \cong S_4$ of order $24$. Under this group, all $6$ optimal solutions form a single orbit (orbit-stabilizer theorem: the stabilizer of any balanced partition has order $|S_4|/6 = 4$). In particular, the flip $\mathbb{Z}_2$ is already contained as a subgroup within the action of $S_4$ on balanced partitions. This notebook addresses the $\mathbb{Z}_2$ flip symmetry; the residual vertex symmetry from $\mathrm{Aut}(K_4)$ after symmetry breaking is discussed as future work.

#### Problem Instance

In [3]:
edges_1   = [(1,2), (1,3), (1,4), (2,3), (2,4), (3,4)]
weights_1 = Dict((u,v) => 1 for (u,v) in edges_1)
N_1       = 4

println("N = ", N_1, ", |E| = ", length(edges_1))
println("Total edge weight = ", sum(values(weights_1)))

N = 4, |E| = 6
Total edge weight = 6


#### Solution Enumeration

In [4]:
# Find optimal value
model_1, z_1, y_1 = build_mc_model(edges_1, weights_1, N_1)
optimize!(model_1)
opt_val_1 = round(Int, objective_value(model_1))

# Enumerate solutions
cuts_baseline_1 = enumerate_cuts(edges_1, weights_1, N_1, opt_val_1,
                                  symmetry_breaking=false)

cuts_reduced_1  = enumerate_cuts(edges_1, weights_1, N_1, opt_val_1,
                                  symmetry_breaking=true)

println("Optimal value = ", opt_val_1)
println("Expected cuts = ", 3 * 2)

println("\nBaseline cuts:")
for (i, (S, Sbar)) in enumerate(cuts_baseline_1)
    println("Cut ", i, ": S = ", S, " | S\u0305 = ", Sbar)
end

println("\nSymmetry-reduced cuts:")
for (i, (S, Sbar)) in enumerate(cuts_reduced_1)
    println("Cut ", i, ": S = ", S, " | S\u0305 = ", Sbar)
end

Optimal value = 4
Expected cuts = 6

Baseline cuts:
Cut 1: S = [3, 4] | S̅ = [1, 2]
Cut 2: S = [1, 4] | S̅ = [2, 3]
Cut 3: S = [1, 2] | S̅ = [3, 4]
Cut 4: S = [2, 4] | S̅ = [1, 3]
Cut 5: S = [2, 3] | S̅ = [1, 4]
Cut 6: S = [1, 3] | S̅ = [2, 4]

Symmetry-reduced cuts:
Cut 1: S = [1, 3] | S̅ = [2, 4]
Cut 2: S = [1, 4] | S̅ = [2, 3]
Cut 3: S = [1, 2] | S̅ = [3, 4]


#### Results Analysis

The optimal cut value is $4$, confirming the analytical bound: a balanced $2$-$2$ partition of $K_4$ cuts exactly $2 \times 2 = 4$ edges out of $6$ total.

Without symmetry breaking, the enumeration identifies all $6$ optimal cuts. They consist of the three essentially distinct balanced partitions together with their flips:

\begin{equation*}
\begin{aligned}
\text{Cut}_1 &: S = \{1,2\},\; \bar{S} = \{3,4\} \\
\text{Cut}_2 &: S = \{1,3\},\; \bar{S} = \{2,4\} \\
\text{Cut}_3 &: S = \{1,4\},\; \bar{S} = \{2,3\} \\
\text{Cut}_4 &: S = \{2,3\},\; \bar{S} = \{1,4\} \\
\text{Cut}_5 &: S = \{2,4\},\; \bar{S} = \{1,3\} \\
\text{Cut}_6 &: S = \{3,4\},\; \bar{S} = \{1,2\}
\end{aligned}
\end{equation*}

The flip pairing is explicit: $\text{Cut}_1 \leftrightarrow \text{Cut}_6$, $\text{Cut}_2 \leftrightarrow \text{Cut}_5$, and $\text{Cut}_3 \leftrightarrow \text{Cut}_4$. Each pair is related by the transformation $z_v \mapsto 1 - z_v$ for all $v \in V$.

Introducing the symmetry-breaking constraint $z_1 = 0$ forces vertex $1$ to the side $S$. This eliminates $\text{Cut}_4$, $\text{Cut}_5$, and $\text{Cut}_6$ (all of which have $z_1 = 1$), retaining the three canonical representatives:

\begin{equation*}
\begin{aligned}
\text{Cut}_1 &: S = \{1,2\},\; \bar{S} = \{3,4\} \\
\text{Cut}_2 &: S = \{1,3\},\; \bar{S} = \{2,4\} \\
\text{Cut}_3 &: S = \{1,4\},\; \bar{S} = \{2,3\}
\end{aligned}
\end{equation*}

The reduction factor is $6 / 3 = 2 = |\mathbb{Z}_2|$, confirming that the flip symmetry was fully broken.

The three remaining solutions exhibit residual symmetry arising from $\mathrm{Aut}(K_4) \cong S_4$. The subgroup of $S_4$ that fixes vertex $1$ is isomorphic to $S_3$ (permutations of $\{2,3,4\}$), with $|S_3| = 6$. Under $S_3$, the three canonical cuts form a single orbit: the transposition $(2,3)$ maps $\text{Cut}_1 \to \text{Cut}_2$, $(2,4)$ maps $\text{Cut}_1 \to \text{Cut}_3$, and so on. The stabilizer of $\text{Cut}_1$ within $S_3$ is $\{\mathrm{id}, (3,4)\}$ of order $2$, giving orbit size $|S_3|/2 = 3$. Complete symmetry breaking would require additional vertex-symmetry constraints and is left as future work.

## **Discussion**

The results obtained in this notebook illustrate the effect of the fixing constraint $z_1 = 0$ on the maximum cut problem formulated as an Integer Linear Program.

In contrast to the shortest path and graph coloring problems, where the dominant symmetry arose from structural properties of the graph or the color-label space, the maximum cut problem exhibits a symmetry that is universal and independent of the instance: the flip $\mathbb{Z}_2$. Because the objective function depends only on which edges cross the cut, and not on which side receives label $0$ or $1$, every feasible solution is always paired with an equivalent complement. This means that without symmetry breaking, every optimal cut is found at least twice regardless of the graph structure or edge weights.

For the complete graph $K_4$ with unit weights, the flip symmetry interacts with the rich vertex symmetry $\mathrm{Aut}(K_4) \cong S_4$. All $6$ optimal solutions form a single orbit under $S_4$ (since $K_4$ is edge-transitive), meaning the flip is already subsumed within the vertex symmetry for this instance. The single fixing constraint $z_1 = 0$ broke the $\mathbb{Z}_2$ flip symmetry exactly, reducing the solution count from $6$ to $3$ and achieving a reduction factor of $2 = |\mathbb{Z}_2|$.

The three remaining canonical solutions are still equivalent under the stabilizer of vertex $1$ in $S_4$, which is $S_3 \cong \mathrm{Sym}(\{2,3,4\})$ with $|S_3| = 6$. They form a single orbit of size $3$ under this subgroup. A complete symmetry-breaking scheme would require additional constraints derived from the orbits of $S_3$ acting on vertex-partition assignments, reducing the solution space to a unique canonical cut. The construction of such hierarchical constraints, as well as the extension of these techniques to the other problems in this repository, constitutes a direction for future work.